In [5]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/benchmarking_paper/datasets")
edges = pd.read_csv(BASE / "pyscenic_net_edges.csv")
rss   = pd.read_csv(BASE / "pyscenic/pruned_rss_score_matrix_final.csv", index_col=0)

# RSS columns look like "ARID3A(+)" — strip the suffix to get bare TF symbol
rss.columns = rss.columns.str.replace(r"\(\+\)$", "", regex=True)

# Compute delRSS = act - rest, per lineage
delrss = pd.DataFrame({
    "delRSS_CD4": rss.loc["CD4_act"] - rss.loc["CD4_rest"],
    "delRSS_CD8": rss.loc["CD8_act"] - rss.loc["CD8_rest"],
})
delrss.index.name = "tf"
delrss = delrss.reset_index()

# Make sure TF column in edges is also stripped of "(+)" if present
edges["tf"] = edges["tf"].str.replace(r"\(\+\)$", "", regex=True)

# Merge delRSS onto every edge by TF
scenic_net = edges.merge(delrss, on="tf", how="left")

# Reorder for clarity: source, target, importance, delRSS_CD4, delRSS_CD8
cols = ["tf", "target", "importance", "delRSS_CD4", "delRSS_CD8"]
scenic_net = scenic_net[[c for c in cols if c in scenic_net.columns]]
scenic_net = scenic_net.rename(columns={"tf": "source"})

# ---- Split into rest vs act networks, per lineage ----
# Convention: delRSS > 0  -> TF more active in activated cells -> ACT network
#             delRSS < 0  -> TF more active in resting cells   -> REST network
# Edges from TFs with delRSS == 0 or NaN are dropped from both (no signal)

def split(df, col):
    act  = df[df[col] >  0].copy()
    rest = df[df[col] <  0].copy()
    return rest, act

cd4_rest, cd4_act = split(scenic_net, "delRSS_CD4")
cd8_rest, cd8_act = split(scenic_net, "delRSS_CD8")

# Save
scenic_net.to_csv(BASE / "scenic_net_full.csv", index=False)
cd4_rest.to_csv(BASE / "scenic_net_CD4_rest.csv", index=False)
cd4_act .to_csv(BASE / "scenic_net_CD4_act.csv",  index=False)
cd8_rest.to_csv(BASE / "scenic_net_CD8_rest.csv", index=False)
cd8_act .to_csv(BASE / "scenic_net_CD8_act.csv",  index=False)

# Sanity check
for name, df in [("CD4 rest", cd4_rest), ("CD4 act", cd4_act),
                 ("CD8 rest", cd8_rest), ("CD8 act", cd8_act)]:
    print(f"{name}: {len(df):>6} edges, {df['source'].nunique():>4} TFs, "
          f"{df['target'].nunique():>5} targets")

CD4 rest:  10886 edges,  154 TFs,  6006 targets
CD4 act:  16133 edges,   81 TFs,  7897 targets
CD8 rest:  18602 edges,  206 TFs,  8031 targets
CD8 act:   8417 edges,   29 TFs,  5393 targets
